In [1]:
!git clone https://github.com/A-Huli/fastcv.git
%cd fastcv

import os

os.environ['PKG_CONFIG_PATH'] = '/usr/lib/x86_64-linux-gnu/pkgconfig'

!apt-get update -qq
!apt-get install -y libopencv-dev
!apt-get install -y $(apt-cache search nsight-systems | grep -oP '^nsight-systems-[0-9\.]+' | head -n 1)


nsys_path = !find /opt/nvidia -name nsys -type f -executable | head -n 1
if nsys_path:
    os.environ['PATH'] += f":{os.path.dirname(nsys_path[0])}"

!pkg-config --modversion opencv4
!nsys --version

Cloning into 'fastcv'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 122 (delta 30), reused 22 (delta 22), pack-reused 82 (from 1)
Receiving objects: 100% (122/122), 4.48 MiB | 11.35 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/fastcv
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas gstreamer1.0-plugins-base
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libavcodec-dev
  libavformat-dev libavutil-dev libcdparanoia0 libcharls2 libdc1394-dev
  libdouble-conversion3 libexif-dev libexif-doc libexif12 libgdcm-dev
  libgdcm3.0 libgl2p

In [3]:
%%shell
cd /content
nvcc -lnvToolsExt -arch=sm_75 -w -O3 main.cu -o main $(pkg-config --cflags --libs opencv4)

nsys profile --trace=cuda,nvtx --stats=true --force-overwrite true -o profil_projektu ./main

Kernel: 0.446236 ms
CUB:   0.2 ms
opencv CPU:    8.82447 ms
Generating '/tmp/nsys-report-1130.qdstrm'
[1/7] [========================100%] profil_projektu.nsys-rep
[2/7] [========================100%] profil_projektu.sqlite
[3/7] Executing 'nvtxsum' stats report

NVTX Range Statistics:

 Time (%)  Total Time (ns)  Instances    Avg (ns)       Med (ns)      Min (ns)     Max (ns)    StdDev (ns)   Style          Range        
 --------  ---------------  ---------  -------------  -------------  -----------  -----------  -----------  -------  --------------------
     99.9      983,796,210          1  983,796,210.0  983,796,210.0  983,796,210  983,796,210          0.0  PushPop  Full Application    
      0.1          744,122          1      744,122.0      744,122.0      744,122      744,122          0.0  PushPop  Histogram Comparison
      0.1          496,671          1      496,671.0      496,671.0      496,671      496,671          0.0  PushPop  LUT Generation      
      0.0          179